# Week 50, Day 5: Hidden states in behaviour: the GLM-HMM

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w050/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Simulating a GLM-HMM

Simulate 3,000 trials: the state switches with probability 1 minus the diagonal of A_true; the stimulus is drawn from nine signed contrasts; the choice is right with probability sigma(w_k . x) for the current state.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2022)
T = 3000
W_true = np.array([[4.0, 0.0],        # engaged: strong stimulus weight, no bias
                   [0.5, 1.5]])       # disengaged: weak stimulus weight, right bias
A_true = np.array([[0.98, 0.02], [0.03, 0.97]])
contrasts = np.array([-1, -0.5, -0.25, -0.125, 0, 0.125, 0.25, 0.5, 1])

s = np.zeros(T, int)
for t in range(1, T):
    ...
stim = r.choice(contrasts, T)
X = np.c_[stim, np.ones(T)]
p_right = ...
y = (r.random(T) < p_right).astype(float)
print(f"fraction of trials engaged {np.mean(s == 0):.2f}; right choices at contrast -1: engaged {y[(stim == -1) & (s == 0)].mean():.2f}, disengaged {y[(stim == -1) & (s == 1)].mean():.2f}, pooled {y[stim == -1].mean():.2f}")
for k, lab in [(0, "engaged"), (1, "disengaged")]:
    plt.plot(contrasts, [y[(stim == c) & (s == k)].mean() for c in contrasts], "o-", label=lab)
plt.plot(contrasts, [y[stim == c].mean() for c in contrasts], "k--", label="pooled")
plt.xlabel("signed contrast"); plt.ylabel("P(right)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_dwell():
    "states persist: mean run length above 20 trials"
    runs = np.diff(np.flatnonzero(np.diff(np.r_[-1, s, -1]) != 0))
    assert runs.mean() > 20
def test_lapse():
    "pooled across states, right choices at the strongest left contrast exceed 10%"
    assert y[stim == -1].mean() > 0.1

test_dwell()
test_lapse()
print("All checks passed")

### Exercise: Posterior over hidden states

Write forward_backward(X, y, W, A, pi): compute the per-trial likelihood of the observed choice under each state, run the scaled forward recursion and the backward recursion, and return gamma, the scaling constants and the forward and backward variables.

In [ ]:
def forward_backward(X, y, W, A, pi):
    P = 1 / (1 + np.exp(-X @ W.T))
    like = np.where(y[:, None] == 1, P, 1 - P)
    T_, K = like.shape
    al = np.zeros((T_, K)); c = np.zeros(T_)
    al[0] = pi * like[0]; c[0] = al[0].sum(); al[0] /= c[0]
    for t in range(1, T_):
        ...
    be = np.ones((T_, K))
    for t in range(T_ - 2, -1, -1):
        ...
    g = al * be; g /= g.sum(1, keepdims=True)
    return g, c, al, be, like

gamma, c, *_ = forward_backward(X, y, W_true, A_true, np.array([0.5, 0.5]))
print(f"state decoding accuracy with true parameters: {np.mean(gamma.argmax(1) == s):.3f}; log-likelihood {np.log(c).sum():.1f}")
plt.plot(gamma[:400, 1], label="P(disengaged)"); plt.plot(s[:400], "k:", label="true state"); plt.xlabel("trial"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_accuracy():
    "with the true parameters, the most probable state matches the true state on more than 85% of trials"
    assert np.mean(gamma.argmax(1) == s) > 0.85

test_accuracy()
print("All checks passed")

### Exercise: Fitting the GLM-HMM

Complete the M-step: accumulate expected transitions xi over trials, normalise rows to get A, and update each state's weights with five Newton steps of gamma-weighted logistic regression. Run 40 iterations from random weights, then order states by stimulus weight.

In [ ]:
def fit_glmhmm(X, y, K=2, iters=40, seed=0):
    g0 = np.random.default_rng(seed)
    W = g0.normal(0, 1, (K, X.shape[1])); A = np.full((K, K), 0.05 / (K - 1)); np.fill_diagonal(A, 0.95); pi = np.ones(K) / K
    for it in range(iters):
        g, c, al, be, like = forward_backward(X, y, W, A, pi)
        xi = np.zeros((K, K))
        for t in range(len(y) - 1):
            xi += al[t][:, None] * A * (like[t + 1] * be[t + 1])[None] / c[t + 1]
        A = ...
        pi = g[0]
        for k in range(K):
            for _ in range(5):
                pk = 1 / (1 + np.exp(-X @ W[k]))
                grad = ...
                H = -(X * (g[:, k] * pk * (1 - pk))[:, None]).T @ X - 1e-3 * np.eye(X.shape[1])
                W[k] -= np.linalg.solve(H, grad)
    return W, A, g, np.log(c).sum()

W_fit, A_fit, g_fit, ll = fit_glmhmm(X, y)
order = np.argsort(-W_fit[:, 0]); W_fit, A_fit, g_fit = W_fit[order], A_fit[np.ix_(order, order)], g_fit[:, order]
print("fitted weights (stimulus, bias):\n", np.round(W_fit, 2)); print("fitted transitions:\n", np.round(A_fit, 3))
print(f"state decoding accuracy {np.mean(g_fit.argmax(1) == s):.3f}; log-likelihood {ll:.1f}")

In [ ]:
import numpy as np
def test_weights():
    "the engaged state's stimulus weight is recovered within 0.8 and the disengaged bias within 0.5"
    assert abs(W_fit[0, 0] - 4) < 0.8 and abs(W_fit[1, 1] - 1.5) < 0.5
def test_states():
    "the fitted model decodes states with accuracy above 0.85"
    assert np.mean(g_fit.argmax(1) == s) > 0.85

test_weights()
test_states()
print("All checks passed")